In [13]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

In [14]:
class State(TypedDict):
    messages: Annotated[list, add_messages]

graph_builder=StateGraph(State)

In [15]:
graph_builder

In [16]:
import os 
from dotenv import load_dotenv
load_dotenv()

True

In [17]:
from langchain_groq import ChatGroq

llm=ChatGroq(model="llama-3.1-8b-instant")

In [18]:
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Llama 3.1 8B Instant', 'release_date': '2024-07-23', 'last_updated': '2024-07-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 131072, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x7f34169be490>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7f34169bee90>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [19]:
## Node functionallity 
def chatbot(state : State):
    return {"messages":[llm.invoke(state["messages"])]}


In [20]:
graph_builder=StateGraph(State)
graph_builder.add_node("llmchatbot", chatbot)

graph_builder.add_edge(START, "llmchatbot")
graph_builder.add_edge("llmchatbot", END)

## Compile graph 
graph=graph_builder.compile()

In [21]:
## visualize the graph 
from pathlib import Path

graph_png = graph.get_graph().draw_mermaid_png()

Path("graph.png").write_bytes(graph_png)



6443

In [24]:
response = graph.invoke({"messages":"Hi"})

In [25]:
response["messages"][-1].content

"It's nice to meet you. Is there something I can help you with or would you like to chat?"

In [28]:
for event in graph.stream({"messages":"Hi how are you"}):
    for value in event.values():
        print(value["messages"][-1].content)

I'm doing well, thank you for asking. I'm a large language model, so I don't have feelings or emotions like humans do, but I'm always happy to chat with you and help with any questions or topics you'd like to discuss. How about you? How's your day going so far?
